# Building a CIM Network Model by Hand

This tutorial builds a small power system network model **from scratch** using
CIMantic Graphs (`cimgraph`). By the end you will have assembled a complete,
exportable CIM model containing every major building block:

| Component | CIM class | Role |
|-----------|-----------|------|
| Grid connection | `EnergySource` | The "slack" / infinite source |
| Lines | `ACLineSegment` | Branches between buses |
| Load | `EnergyConsumer` | A customer load |
| Switch | `LoadBreakSwitch` | Sectionalizing / isolation device |
| Capacitor | `LinearShuntCompensator` | Shunt capacitor for voltage support |
| Battery inverter | `PowerElectronicsConnection` + `BatteryUnit` | A DER |
| Buses | `ConnectivityNode` | Electrical nodes |
| Connections | `Terminal` | How equipment attaches to buses |
| Containers | `Feeder`, `Substation` | Organize and name the model |

### The network we will build

A 5-bus radial feeder: the grid feeds `bus_1`; three line segments carry power
down to `bus_4`; a load taps off `bus_2`; a shunt capacitor supports voltage at
`bus_3`; and a `LoadBreakSwitch` connects `bus_4` to `bus_5`, where a
battery-backed inverter sits. The switch lets us isolate the DER from the rest of
the feeder.

In [ ]:
from pathlib import Path

# Start from a clean slate: remove any stale model file from a previous run.
# cimgraph loads an existing file into the graph on connect, and because objects
# use deterministic UUIDs, a pre-existing (possibly out-of-sync) object would be
# kept over the one we build here — leaving equipment without its terminals.
Path('hand_built_model.xml').unlink(missing_ok=True)

## 1. Loading a CIM profile

A CIM **profile** is a data dictionary that defines the set of classes, attributes, and associations available in a model. CIMantic Graphs ships several pre-built profiles, each generated directly from a CIM UML schema.

Importing a profile module gives you access to its dataclasses (e.g. `cim.EnergyConsumer`, `cim.ACLineSegment`). Setting the `CIMG_CIM_PROFILE` environment variable tells the library which profile to use when serializing and parsing models.

Here we load `cimhub_2026' - a profile based on CIM18v15 with distribution classes

In [ ]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2026'
os.environ['CIMG_USE_UNITS'] = 'TRUE'   # enable pint-backed CIMUnit values

import cimgraph.data_profile.cimhub_2026 as cim

Because each profile is just a set of Python dataclasses, you can inspect any class directly. CIMantic Graphs provides `get_mermaid()` to render a class and its attributes as a [Mermaid](https://mermaid.js.org/) class diagram, making it easy to see the structure of a CIM class at a glance.

Here we visualize `EnergyConsumer`, the CIM class representing a point of energy consumption (a load) on the power system.

In [ ]:
from mermaid import Mermaid
from cimgraph import utils
Mermaid(utils.get_mermaid(cim.EnergyConsumer))

Each profile class also carries its CIM documentation as a Python docstring, so you can read the official description of any class directly:

In [ ]:
print(cim.EnergyConsumer.__doc__)

## 2. Create the network graph

Everything we build must be registered in a **graph** so it can be queried and exported. The graph is backed by a connection, which can either be a database (such as Neo4J) or a file. For this example, we will use an `XMLFile` that we will write to at the end.

> The two warnings below are expected: the file doesn't exist yet, so cimgraph starts from an empty graph.

In [ ]:
from cimgraph.models import FeederModel
from cimgraph.databases import XMLFile

In [ ]:
file = XMLFile('hand_built_model.xml')
network = FeederModel(container=None, connection=file)

## 2. Identifiers and Naming

Almost all classes within CIM inherit from the top-level abstract class `IdentifiedObject`, which a set of attributes for specifying the name, description, and primary key used to track the object in the database. 

One of the most important attributes of IdentifiedObject is the master resource identifier (mRID), which is a globally unique 3- to 18-character identifier of objects; the mRID does not have to be human-readable. This identifier is generally intended to be used by software systems and is never displayed to the user. In CIM-based applications, the mRID is often used as the unique pointer or dictionary key that allows all the properties of a particular piece of equipment to be called from memory or queried from a database.

To enforce global uniqueness and consisting formatting of identifiers used as the primary keys within the knowledge graph dictionary of CIMantic Graphs, all classes are forced to inherit from the new CIM18 `Identity`, which uses `Identity.identifier` as the primary key for tracking objects. Unlike mRID, which is a string, the identifier attribute is strictly typed to be a UUID 128-bit integer. It is also used as the RDF ID in serialization exchanges

In [ ]:
diagram_text = utils.get_mermaid([cim.Identity, cim.IdentifiedObject])
Mermaid(diagram_text)

When new objects are created using CIMantic Graphs, the internal `.uuid()` method generates a valid UUID upon object creation

In [ ]:
new_line = cim.ACLineSegment()
new_line.pprint()

By default each object gets a random UUID. But identifiers can also be **deterministic**: given the same name and seed (domain URI), CIMantic Graphs generates the same UUID every time. This is what makes the two objects below compare as equal even though they were created independently.

In [ ]:
line1 = cim.ACLineSegment(name='my_line', mRID = 'local_id_12345')
line2 = cim.ACLineSegment(name='my_line', mRID = 'local_id_12345')
print(line1)
print(line2)
print(line1 == line2)

## 3. Equipment Containers

For the purposes of naming, asset management, and network modeling, all nodes and equipment are associated with a container class. Each node will have a container as specified by the `ConnectivityNode.ConnectivityNodeContainer` association, while equipment will have a primary container specified by `Equipment.EquipmentContainer`.

For a transmission models, equipment is sorted into a `Substation`, `VoltageLevel`, and `Line`.

For a distribution model the top-level container is a `Feeder`, energized by a `Substation`.

In [ ]:
diagram_text = utils.get_mermaid_path(cim.Substation,'NormalEnergizedFeeder')
diagram_text = utils.add_mermaid_path(cim.Feeder, 'NormalEnergizingSubstation', diagram_text)
Mermaid(diagram_text)

We are going to create a demo substation and feeder, after which we add them to the network model graph. All objects need to be added to the graph for them to be serialized in model export.

In [ ]:
feeder = cim.Feeder(name='demo_feeder')
substation = cim.Substation(name='demo_sub')

network.add_to_graph(feeder)
network.add_to_graph(substation)

All the CIM objects added to graph are available through `network.graph`, which is a two-level dictionary sorted by class type (`cim.ClassName`) and UUID. The objects can also be retrieved using the `network.list_by_class(cim.ClassName)`

In [ ]:
network.graph[cim.Feeder]

In [ ]:
for obj in network.list_by_class(cim.Feeder):
    print(feeder.name)

We link objects together through direct in-memory pointers based on the UML association name

In [ ]:
feeder.NormalEnergizingSubstation = substation
feeder.pprint()

CIM-Graph also supports reverse associations, which are not serialized in the XML exchange but are very useful for runtime graph parsing. CIM-Graph auto-populates the reverse associations when reading from a file or database for ease of use.

In [ ]:
substation.NormalEnergizedFeeder = feeder # reverse association
substation.pprint()

## 4. Base Voltage

The `BaseVoltage` class is used to denote the as-operated nominal voltage of equipment within the model. All equipment of the same nominal voltage are associated with a single object instance.

For this demo, we are going to build a basic 12.47kV distribution feeder, so we define a new object and set the `BaseVoltage.nominalVoltage` value to 12.47kV.

Notably, CIM-Graph uses the core CIM-Datatype classes, which are internally mapped to python pint quantities for automated unit conversion.

In [ ]:
base_12kv = cim.BaseVoltage(name='base_12.47kV')
base_12kv.nominalVoltage = cim.Voltage(12.47, 'kV')
network.add_to_graph(base_12kv)

print(base_12kv.nominalVoltage)            # stored in volts
print(base_12kv.nominalVoltage.to('kV'))   # convert back for display

## 5. Nodes and Terminals

All current-carrying electrical equipment inheriting from the `ConductingEquipment` class are defined as being connected to a `Terminal` object associated with a `ConnectivityNode`. 

In [ ]:
diagram = utils.get_mermaid([
    cim.ConductingEquipment,
    cim.Terminal,
    cim.ConnectivityNode,
], show_attributes=False)
Mermaid(diagram)

Generators, shunt capacitors, shunt reactors, and loads are connected to one Terminal object associated with a single `ConnectivityNode`. 

Lines, cables, series capacitors, and series reactors have two Terminal objects associated with either end of the branch. 



![Alt text](./images/Fig_8_Chapter_5.png)

Rather than defining from/to buses, CIM identifies the end of a branch by setting the `ACDCTerminal.sequenceNumber` attribute of a terminal to values of 1 or 2 to specify the particular end. 

Transformers are defined using two or three terminals, with the `ACDCTerminal`. `sequenceNumber` attribute used to designate whether the Terminal is associated with the primary, secondary, or tertiary windings. Split-phase distribution transformers also use three terminals, but the two low-side terminals are associated with a single `ConnectivityNode`.

In [ ]:
bus_names = ['bus_1', 'bus_2', 'bus_3', 'bus_4', 'bus_5']

for name in bus_names:
    node = cim.ConnectivityNode(name=name, ConnectivityNodeContainer=feeder)
    network.add_to_graph(node)

for node in network.list_by_class(cim.ConnectivityNode):
    print(node)

### Helper Functions

CIM-Graph allows creation of simple helper functions to automate repetitive tasks, such as connecting equipment to a bus. 

1. Look up the correct CIM `ConnetivityNode` based on the bus name

2. Create a `Terminal` with a `sequenceNumber` (1 for the first/only end, 2 for
   the far end of a branch).
3. Point the terminal at its equipment (`ConductingEquipment`) and its bus
   (`ConnectivityNode`).
4. Append it to both reverse lists so the graph is navigable in both directions.


In [ ]:
def connect(equipment, bus_name, sequence_number):
    """Attach `equipment` to `bus` via a new Terminal and register it."""
    
    node = network.find_by_attribute(cim.ConnectivityNode, 'name', bus_name)[0]
    
    terminal = cim.Terminal(
        name=f'{equipment.name}_t{sequence_number}',
        sequenceNumber=sequence_number,
    )
    terminal.ConductingEquipment = equipment
    terminal.ConnectivityNode = node
    equipment.Terminals.append(terminal)
    node.Terminals.append(terminal)
    network.add_to_graph(terminal)
    return terminal

## 7. EnergySource — the grid connection

The `EnergySource` represents the connection to the bulk grid (the "slack bus"
or infinite source). It connects to `bus_1` with a single terminal.

In [ ]:
source = cim.EnergySource(name='grid', EquipmentContainer=feeder)
source.BaseVoltage = base_12kv
source.nominalVoltage = cim.Voltage(12.47, 'kV')
network.add_to_graph(source)

connect(source, 'bus_1', sequence_number=1)

source.pprint()

## 8. ACLineSegments — the branches

Lines connect two buses, so each `ACLineSegment` gets **two** terminals
(sequenceNumber 1 and 2). We define impedance directly with the positive-sequence
`r`/`x` attributes (in ohms) and a `length`.

We build the three line segments with a small loop.

In [ ]:
line_specs = [
    # (name,       from_bus, to_bus,  length_ft, r_ohm,  x_ohm)
    # Impedances ~ 0.30 + j0.60 ohm/mile (typical 4/0 ACSR overhead) x length
    ('line_1_2',  'bus_1',  'bus_2',  1000,      0.0568, 0.1136),
    ('line_2_3',  'bus_2',  'bus_3',   800,      0.0455, 0.0909),
    ('line_3_4',  'bus_3',  'bus_4',   500,      0.0284, 0.0568),
]

lines = {}
for name, from_bus, to_bus, length_ft, r_ohm, x_ohm in line_specs:
    line = cim.ACLineSegment(name=name, EquipmentContainer=feeder)
    line.BaseVoltage = base_12kv
    line.length = cim.Length(length_ft, 'ft')
    line.r = cim.Resistance(r_ohm, 'ohm')
    line.x = cim.Reactance(x_ohm, 'ohm')

    network.add_to_graph(line)
    connect(line, from_bus, sequence_number=1)
    connect(line, to_bus,   sequence_number=2)
    lines[name] = line


In [ ]:
Mermaid(utils.get_mermaid(line))

Notice the line now reports a `length` in meters (SI) even though we entered
feet — that's the `CIMUnit` system converting on input.

## 9. Recloser

A switch connects two buses, so like a line it has **two** terminals. CIM has
several switch types (`Breaker`, `Recloser`, `Fuse`, ...); we use a
`Recloser`, common on distribution feeders. Its state is set by two
booleans:

- `normalOpen` — the switch's *normal* position (open or closed)
- `open` — its *current* position in this study


In [ ]:
Mermaid(utils.get_mermaid([cim.Switch, cim.ProtectedSwitch, cim.Recloser]))


We place it between `bus_4` and `bus_5`, closed, so power can reach the battery
inverter on `bus_5`. Opening it would isolate the DER.

In [ ]:
switch = cim.Recloser(name='sw_4_5', EquipmentContainer=feeder)
switch.BaseVoltage = base_12kv
switch.normalOpen = False    # normally closed
switch.open = False          # closed in this study
switch.ratedCurrent = cim.CurrentFlow(600, 'A')
network.add_to_graph(switch)

connect(switch, 'bus_4', sequence_number=1)
connect(switch, 'bus_5', sequence_number=2)

In [ ]:
Mermaid(utils.get_mermaid(switch))

## 10. EnergyConsumer — the load

A load is a single-terminal device. We attach it to `bus_2` and set its real and
reactive power demand. CIM's terminal sign convention: positive `p` is power
flowing **into** the equipment, so a consuming load has positive `p`.

In [ ]:
load = cim.EnergyConsumer(name='load_a', EquipmentContainer=feeder)
load.BaseVoltage = base_12kv
load.p = cim.ActivePower(0.5, 'MW')
load.q = cim.ReactivePower(0.15, 'MVAr')
network.add_to_graph(load)

connect(load, 'bus_2', sequence_number=1)

In [ ]:
Mermaid(utils.get_mermaid(load))

## 11. LinearShuntCompensator — a shunt capacitor

A shunt capacitor injects reactive power to support voltage. It is a
single-terminal (shunt) device. CIM specifies the capacitor by its
**susceptance per section** (`bPerSection`, in siemens) rather than directly in
kvar — the relationship is:

$$ b = \frac{Q_{kvar} / 1000}{V_{kV}^2 \times N_{sections}} $$

We size a 150 kvar bank at `bus_3` (one section) to lift the mid-feeder voltage.

In [ ]:
kvar_target = 150.0
kv = 12.47
b_per_section = 0.001 * kvar_target / (kv ** 2)   # siemens, for one section

capacitor = cim.LinearShuntCompensator(name='cap_3', EquipmentContainer=feeder)
capacitor.BaseVoltage = base_12kv
capacitor.nomU = cim.Voltage(12.47, 'kV')
capacitor.bPerSection = cim.Susceptance(b_per_section, 'S')
capacitor.maximumSections = 1
capacitor.sections = 1                              # one section in service
capacitor.phaseConnection = cim.PhaseShuntConnectionKind.Y
network.add_to_graph(capacitor)

connect(capacitor, 'bus_3', sequence_number=1)

## 12. Battery inverter — PowerElectronicsConnection + BatteryUnit

A battery-backed inverter is modeled in two parts:

- **`PowerElectronicsConnection`** (PEC) — the inverter itself. It is the
  `ConductingEquipment` that connects to the AC network with a single terminal,
  and carries the AC ratings (`ratedS`, `ratedU`, `maxQ`/`minQ`).
- **`BatteryUnit`** — the DC source behind the inverter. It carries the energy
  storage attributes (`ratedE`, `storedE`, `batteryState`) and is associated to
  the PEC via `PowerElectronicsUnit`.

The DC side has no terminals — only the inverter connects to the AC grid.

In [ ]:
# The inverter (AC side)
inverter = cim.PowerElectronicsConnection(name='battery_inverter', EquipmentContainer=feeder)
inverter.BaseVoltage = base_12kv
inverter.ratedS = cim.ApparentPower(0.25, 'MVA')
inverter.ratedU = cim.Voltage(12.47, 'kV')
inverter.maxQ = cim.ReactivePower(0.10, 'MVAr')
inverter.minQ = cim.ReactivePower(-0.10, 'MVAr')
network.add_to_graph(inverter)

# Connect the inverter to bus_5 (beyond the switch)
connect(inverter, 'bus_5', sequence_number=1)

# The battery (DC side)
battery = cim.BatteryUnit(name='battery_a')
battery.ratedE = cim.RealEnergy(0.5, 'MWh')      # capacity
battery.storedE = cim.RealEnergy(0.3, 'MWh')     # current state of charge
battery.maxP = cim.ActivePower(0.25, 'MW')
battery.minP = cim.ActivePower(-0.25, 'MW')
battery.batteryState = cim.BatteryStateKind.discharging
network.add_to_graph(battery)

# Associate the battery with its inverter (both directions)
battery.PowerElectronicsConnection = inverter
inverter.PowerElectronicsUnit.append(battery)


In [ ]:
Mermaid(utils.get_mermaid(inverter))

We can read the battery's state of charge by walking from the inverter to its
unit — the same traversal an analysis tool would use.

In [ ]:
for unit in inverter.PowerElectronicsUnit:
    if isinstance(unit, cim.BatteryUnit):
        soc = float(unit.storedE) / float(unit.ratedE)
        print(f'{unit.name}: state of charge = {soc:.0%}')

## 13. Inspect the assembled model

The graph is keyed by class. We can list how many of each object we created.

In [ ]:
for cim_class in [cim.Feeder, cim.Substation, cim.BaseVoltage,
                  cim.ConnectivityNode, cim.Terminal, cim.ACLineSegment,
                  cim.Recloser, cim.LinearShuntCompensator,
                  cim.EnergySource, cim.EnergyConsumer,
                  cim.PowerElectronicsConnection, cim.BatteryUnit]:
    count = len(network.graph.get(cim_class, {}))
    print(f'{count:>2}  {cim_class.__name__}')

## 14. Visualize the actual objects

The UML diagrams earlier showed the *class* relationships. Now we render the
**actual instances** we built and their links. `get_mermaid_path()` starts a
diagram from a root object and follows an attribute path; `add_mermaid_path()`
grafts on additional branches.

First, line `line_1_2` with both of its terminals, the buses they connect to,
and its containing feeder:

In [ ]:
line = lines['line_1_2']

diagram = utils.get_mermaid_path(line, ['Terminals', '[0]', 'ConnectivityNode'])
diagram = utils.add_mermaid_path(line, ['Terminals', '[1]', 'ConnectivityNode'], diagram)
diagram = utils.add_mermaid_path(line, 'EquipmentContainer', diagram)
Mermaid(diagram)

The switch `sw_4_5` looks structurally like a line — two terminals bridging
`bus_4` and `bus_5`:

In [ ]:
diagram = utils.get_mermaid_path(switch, ['Terminals', '[0]', 'ConnectivityNode'])
diagram = utils.add_mermaid_path(switch, ['Terminals', '[1]', 'ConnectivityNode'], diagram)
Mermaid(diagram)

The battery inverter — showing both the DC side (`PowerElectronicsUnit` →
`BatteryUnit`) and the AC side (`Terminals` → `ConnectivityNode` at `bus_5`):

In [ ]:
diagram = utils.get_mermaid_path(inverter, ['PowerElectronicsUnit', '[0]'])
diagram = utils.add_mermaid_path(inverter, ['Terminals', '[0]', 'ConnectivityNode'], diagram)
Mermaid(diagram)

Finally, a bus-centric view: `bus_2` and every piece of equipment attached to
it (the two line segments and the load):

In [ ]:
bus_2 = network.find_by_attribute(cim.ConnectivityNode,'name','bus_2')[0]

diagram = utils.get_mermaid_path(bus_2, ['Terminals', '[0]', 'ConductingEquipment'])
for i in range(1, len(bus_2.Terminals)):
    diagram = utils.add_mermaid_path(
        bus_2, ['Terminals', f'[{i}]', 'ConductingEquipment'], diagram)
Mermaid(diagram)

## 15. Export to CIM XML

Finally, write the whole graph out as a CIM RDF/XML file. We use
`cimgraph.utils.write_xml()`, which serializes every object in the graph.

The CIM profile spreads attributes across several **namespaces** (the core
`cim` namespace plus a handful of extensions). `write_xml` needs the prefix for
each namespace it encounters, so we pass the full set explicitly.

In [ ]:
namespaces = {
    'cim':'http://cim.ucaiug.io/CIM101/draft#',
    'rdf':'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
    'eu': 'http://iec.ch/TC57/CIM100-European#',
    'nc': 'http://entsoe.eu/ns/nc#',
    'gb': 'http://GB/placeholder/ext#',
    'gad':'http://gridappsd.org/CIM/extension#',
}

utils.write_xml(network, 'hand_built_model.xml', namespaces=namespaces)
print('Wrote hand_built_model.xml')

### Read it back into a new model

Now load the file we just wrote into a **fresh** `FeederModel` and confirm the
objects survived the round trip. We compare object counts per class against the
model we built in memory.

In [ ]:
reloaded = FeederModel(connection=XMLFile('hand_built_model.xml'), container=None)

In [ ]:
for cim_class in [cim.Feeder, cim.Substation, cim.BaseVoltage,
                  cim.ConnectivityNode, cim.Terminal, cim.ACLineSegment,
                  cim.EnergySource, cim.EnergyConsumer, cim.Recloser,
                  cim.LinearShuntCompensator,
                  cim.PowerElectronicsConnection, cim.BatteryUnit]:
    built = len(network.graph.get(cim_class, {}))
    loaded = len(reloaded.graph.get(cim_class, {}))
    status = 'ok' if built == loaded else 'MISMATCH'
    print(f'{cim_class.__name__:>28}: built={built:>2}  reloaded={loaded:>2}  [{status}]')

And confirm the attribute values came through — list the reloaded lines with
their impedance and length:

In [ ]:
for line in reloaded.list_by_class(cim.ACLineSegment):
    print(f'{line.name}: length = {float(line.length):.1f} m, '
          f'r = {float(line.r):.2f} ohm, x = {float(line.x):.2f} ohm')

## 16. Export to OpenDSS and solve a power flow

A CIM model describes the *network*; to compute voltages and power flows we hand
it to an analysis engine. Here we export to **OpenDSS** using the CIMHub
`cim_to_dss` exporter, then solve with `opendssdirect`.

> **Optional section.** This needs two extra packages: `cimhub-opendss` (the
> exporter) and `opendssdirect-py` (the solver). They are **not** installed in
> the default codespace environment (CIMHub is not yet published to PyPI). The
> cells below detect this and skip cleanly if the packages are missing — see the
> README for how to enable OpenDSS export. Everything earlier in this notebook
> runs without them.

In [ ]:
from pathlib import Path
import tempfile

from cimhub_opendss.exporter.cim_to_dss import cim_to_dss
import opendssdirect as dss

### Export the feeder to DSS files

`cim_to_dss(network, output_dir)` writes a set of `.dss` files — one per
equipment category — plus a `Master.dss` that ties them together. It returns a
dict of `{filename: [commands]}` so we can see exactly what was generated.

In [ ]:
output_dir = Path(tempfile.mkdtemp(prefix='demo_dss_'))
dss_files = cim_to_dss(network, output_dir)

for filename, commands in dss_files.items():
    print(f'### {filename}')
    for command in commands:
        print(f'  {command}')
    print()

Each CIM object became its native DSS element:

| CIM | DSS |
|-----|-----|
| `EnergySource` | `Circuit` / `Vsource` |
| `ACLineSegment` | `Line` |
| `LoadBreakSwitch` | `Line` (`Switch=Yes`) |
| `EnergyConsumer` | `Load` |
| `LinearShuntCompensator` | `Capacitor` |
| `PowerElectronicsConnection` + `BatteryUnit` | `Storage` |

### Solve the power flow

Compile the generated `Master.dss` and solve.

In [ ]:
# Solve with OpenDSS-Direct
master = output_dir / 'Master.dss'

dss.Text.Command('Clear')
dss.Text.Command(f'Compile [{master}]')
dss.Text.Command('Solve')

print('Converged:', dss.Solution.Converged())
total_p, total_q = dss.Circuit.TotalPower()  # power into the circuit (kW, kVAr)
print(f'Total source power: {-total_p:.1f} kW, {-total_q:.1f} kVAr')
print(f'Buses: {dss.Circuit.NumBuses()}, Elements: {dss.Circuit.NumCktElements()}')

### Bus voltages

Read the per-unit voltage magnitude at every bus. A healthy distribution feeder
sits near 1.0 pu, sagging slightly toward the end of the line under load.

In [ ]:
print(f'{"Bus":8s}  {"V (pu)":>8s}')
print('-' * 20)
for name in dss.Circuit.AllBusNames():
    dss.Circuit.SetActiveBus(name)
    v_pu = dss.Bus.puVmagAngle()[0]   # magnitude of the first phase
    print(f'{name:8s}  {v_pu:8.4f}')

### Element power flows

The active power at the first terminal of each element. By OpenDSS convention,
power flowing **out of** an element into the circuit is negative — so the source
(`Vsource`) and the discharging battery (`Storage`) show negative P, while the
load (`Load`) draws positive P. The load is `model=1` (constant power), so it
pulls its full 500 kW regardless of the voltage sag. The closed switch
(`Line.sw_4_5`, zero impedance) and the reactive-only capacitor
(`Capacitor.cap_3`) both show ~0 kW.

In [ ]:
print(f'{"Element":24s}  {"P (kW)":>10s}')
print('-' * 38)
for name in dss.Circuit.AllElementNames():
    dss.Circuit.SetActiveElement(name)
    powers = dss.CktElement.Powers()          # [P1, Q1, P2, Q2, ...] per phase
    n_phases = dss.CktElement.NumPhases()
    p_total = sum(powers[i * 2] for i in range(n_phases))
    print(f'{name:24s}  {p_total:10.2f}')

## Summary

You built a complete CIM network model by hand:

1. **Set the profile** and created a graph backed by an `XMLFile`.
2. **Containers** (`Feeder`, `Substation`) to organize and name equipment.
3. A shared **`BaseVoltage`** using `CIMUnit` (never hand-converting units).
4. **Buses** as `ConnectivityNode` objects.
5. A **`connect()` helper** encapsulating the Terminal wiring pattern.
6. An **`EnergySource`** grid connection, three **`ACLineSegment`** branches, a
   **`LoadBreakSwitch`**, an **`EnergyConsumer`** load, a
   **`LinearShuntCompensator`** capacitor, and a
   **`PowerElectronicsConnection` + `BatteryUnit`** inverter.
7. **Visualized** both the class hierarchy (`get_mermaid`) and the actual object
   instances (`get_mermaid_path` / `add_mermaid_path`).
8. **Exported** to CIM XML with `utils.write_xml()` and **reloaded** into a new
   `FeederModel` to verify.
9. **Exported to OpenDSS** with `cim_to_dss`, **solved** the power flow, and read
   back bus voltages and element power flows.

### Key takeaways
- Every object must be added with `network.add_to_graph(...)`.
- Equipment connects to buses through `Terminal` objects; branches use two
  terminals (sequenceNumber 1 and 2), shunt devices use one.
- Use `CIMUnit` constructors (`cim.Voltage`, `cim.ActivePower`, ...) on input and
  `.to(unit)` on output — never multiply by `1e3`/`1e6` by hand.
- Associations are directional; set the reverse side when you need to traverse
  both ways.